# Belgium Job–Home Network — Master Data Preparation v1

This notebook consolidates the Belgian job–home data preparation pipeline into one reproducible workflow. It produces two parallel analytical data systems: (A) municipality × year and (B) origin × destination × year.

Design principles: 2025 565-municipality geography is canonical; missing/suppressed OD cells are never silently converted to zero; residence-side and workplace-side quantities remain separate; post-treatment variables are labelled rather than automatically used as controls; the full VAR direct-download archive is indexed for exploration without blindly cross-joining marginal tables.

Default paths match the project at D:\OneDrive - Universiteit Utrecht\31_BL_Network. The notebook does not run DID models; it prepares auditable inputs for municipality DID/event-study/spatial models and edge-level PPML/network models.

## 0. Configuration

In [ ]:

from __future__ import annotations
from pathlib import Path
from collections import defaultdict
import csv, gzip, hashlib, json, math, re, unicodedata, warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

ROOT = Path(r"D:\OneDrive - Universiteit Utrecht\31_BL_Network")
RAW = ROOT / "raw_data"
OUT = ROOT / "02_output" / "master_data_prep_v1"
SUPPORT = OUT / "support"
QA_DIR = OUT / "qa"
for p in (OUT, SUPPORT, QA_DIR):
    p.mkdir(parents=True, exist_ok=True)

YEAR_MIN, YEAR_MAX = 2015, 2024
YEARS = list(range(YEAR_MIN, YEAR_MAX + 1))
PRE_YEARS = list(range(2015, 2020))
EXPECTED_N = 565
BASELINE_YEAR = 2019
SHOCK_YEAR = 2020

# Core geography and existing source products
BASEMAP = RAW / "Belgium_VAR_Basemap_2025" / "Belgium_VAR_565_2025.geojson"
POP_XLSX = RAW / "controls" / "Bevolking_per_gemeente.xlsx"
POP_PANEL_EXISTING = RAW / "controls" / "population_density_565" / "Belgium_Population_Area_Density_565_2015_2024.csv"

HOUSE_XLSX = RAW / "controls" / "vastgoed_2010_9999.xlsx"
TAX_XLSX = RAW / "controls" / "TF_PSNL_INC_TAX_MUNTY.xlsx"
ADI_XLSX = RAW / "controls" / "TF_SOC_ADI_MUNTY.xlsx"
CONTROL_PANEL_EXISTING = ROOT / "02_output" / "controls_565" / "municipality_controls_565_2015_2024.parquet"
CONTROL_PANEL_EXISTING_CSV = ROOT / "02_output" / "controls_565" / "municipality_controls_565_2015_2024.csv"

OD_DIR = RAW / "VAR_Belgium_2015_2024" / "02_processed"
OD_FILES = {
    "total": OD_DIR / "od_main_2015_2024_age20_64.csv",
    "age": OD_DIR / "od_by_age_2015_2024.csv",
    "sex": OD_DIR / "od_by_sex_2015_2024_age20_64.csv",
    "education": OD_DIR / "od_by_education_2015_2024_age20_64.csv",
    "employment_status": OD_DIR / "od_by_status_2015_2024_age20_64.csv",
    "sector": OD_DIR / "od_by_sector_2015_2024_age20_64.csv",
}

# Full direct VAR archive collected in the previous step
VAR_DIRECT_ROOT = RAW / "VAR_CSV_Direct_v2"

# Telework components used in the earlier Belgian DID work
TELEWORK_XLSX = RAW / "telework_exposure" / "Telework_LFS_STATBEL_nl.xlsx"
TELEWORK_BUILDER = ROOT / "02_output" / "telework_exposure_v4"
WFH_RATE_CSV = TELEWORK_BUILDER / "01_nace_year_wfh_rates_2010_2025.csv"
WFH_WORKPLACE_MARGIN_CSV = TELEWORK_BUILDER / "04_var_published_workplace_margins.csv"

# Runtime choices
PREFER_EXISTING_POP_PANEL = True
PREFER_EXISTING_SOCIO_CONTROL_PANEL = True
BUILD_FULL_VAR_CATALOG = True
MATERIALIZE_FULL_VAR_VIEWS = False  # can be huge; inventory is built regardless
CREATE_BALANCED_PAIR_SKELETON = False  # 565^2 * 10 = 3,192,250 rows
TREAT_UNPUBLISHED_OD_AS_ZERO = False  # intentionally False: suppression/missingness is not a verified zero
WRITE_MUNICIPALITY_CSV = True
WRITE_OD_CSV_GZ = False
WRITE_GEOJSON_2024 = True

print("Project:", ROOT)
print("Output:", OUT)


## 1. Helpers and the canonical 2025 municipality crosswalk

In [ ]:

def require(condition, message):
    if not bool(condition):
        raise ValueError(message)

def nis5(x):
    if pd.isna(x):
        return np.nan
    s = re.sub(r"\.0$", "", str(x).strip())
    return s.zfill(5) if s.isdigit() else s

def norm(x):
    if pd.isna(x):
        return ""
    return " ".join(str(x).replace("\xa0", " ").strip().split()).casefold()

def ascii_slug(x):
    s = unicodedata.normalize("NFKD", str(x)).encode("ascii", "ignore").decode().lower()
    return re.sub(r"[^a-z0-9]+", "_", s).strip("_")

def safe_div(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    return np.divide(a, b, out=np.full(np.broadcast(a,b).shape, np.nan), where=np.isfinite(a)&np.isfinite(b)&(b!=0))

def write_table(df, path, csv_also=False):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.suffix.lower() == ".parquet":
        df.to_parquet(path, index=False)
        if csv_also:
            df.to_csv(path.with_suffix(".csv"), index=False, encoding="utf-8-sig")
    else:
        df.to_csv(path, index=False, encoding="utf-8-sig")
    return path

def weighted_quantile(values, weights, probs=(.5,.75,.9)):
    v = np.asarray(values, float)
    w = np.asarray(weights, float)
    keep = np.isfinite(v) & np.isfinite(w) & (w > 0)
    if not keep.any():
        return np.full(len(probs), np.nan)
    v, w = v[keep], w[keep]
    order = np.argsort(v, kind="stable")
    v, w = v[order], w[order]
    cw = np.cumsum(w)
    targets = np.asarray(probs) * cw[-1]
    return v[np.minimum(np.searchsorted(cw, targets, side="left"), len(v)-1)]

MERGERS_2019 = {
    "72042": ["72040", "71047"], "72043": ["72025", "72029"],
    "45068": ["45017", "45057"], "44084": ["44001", "44029"],
    "44083": ["44011", "44049"], "12041": ["12030", "12034"],
    "44085": ["44072", "44036", "44080"],
}
RECODE_2019 = {
    "55022": "58001", "56011": "58002", "56085": "58003", "56087": "58004",
    "52063": "55085", "52043": "55086", "55010": "51067", "55039": "51068",
    "55023": "51069", "54007": "57096", "54010": "57097",
}
MERGERS_2025 = {
    "11002": ["11002", "11007"], "23106": ["23023", "23024", "23032"],
    "37021": ["37012", "37018"], "37022": ["37007", "37015"],
    "44086": ["44012", "44048"], "44087": ["44034", "44073"],
    "46029": ["46014", "44045"], "44088": ["44040", "44043"],
    "46030": ["46003", "46013", "11056"], "73110": ["73006", "73032"],
    "73111": ["73009", "73083"], "71071": ["71069", "71057"],
    "71072": ["71022", "73040"], "82039": ["82003", "82005"],
}

FINAL_RECODE = {}
for target, sources in MERGERS_2019.items():
    for source in sources:
        FINAL_RECODE[source] = target
FINAL_RECODE.update(RECODE_2019)
for target, sources in MERGERS_2025.items():
    for source in sources:
        FINAL_RECODE[source] = target

def to_2025_nis(x):
    c = nis5(x)
    if pd.isna(c):
        return np.nan
    seen = set()
    while c in FINAL_RECODE and c not in seen:
        seen.add(c)
        nxt = FINAL_RECODE[c]
        if nxt == c:
            break
        c = nxt
    return c

def merge_unique(base, table, keys=("nis","year"), label="table"):
    if table is None or len(table) == 0:
        return base
    table = table.copy()
    keys = list(keys)
    require(not table.duplicated(keys).any(), f"{label}: duplicate keys {keys}")
    overlap = [c for c in table.columns if c in base.columns and c not in keys]
    if overlap:
        table = table.drop(columns=overlap)
    return base.merge(table, on=keys, how="left", validate="one_to_one")


## 2. Canonical geography, area, pair distance and contiguity

In [ ]:

require(BASEMAP.exists(), f"Missing basemap: {BASEMAP}")
geo = gpd.read_file(BASEMAP).copy()
geo["nis"] = geo["nis"].map(nis5)
require(len(geo) == EXPECTED_N, f"Expected {EXPECTED_N} municipalities; found {len(geo)}")
require(geo["nis"].is_unique, "Duplicate NIS codes in basemap")
require(geo["node_id"].is_unique, "Duplicate VAR node_id in basemap")

geo_area = geo.to_crs(6933)
geo["area_km2"] = geo_area.geometry.area / 1e6
geo["ln_area_km2"] = np.log(geo["area_km2"])
geo["longitude"] = pd.to_numeric(geo["flow_longitude"], errors="coerce")
geo["latitude"] = pd.to_numeric(geo["flow_latitude"], errors="coerce")
require(geo[["longitude","latitude"]].notna().all().all(), "Missing flow coordinates")

ids = geo["nis"].to_numpy()
lon = np.radians(geo["longitude"].to_numpy(float))
lat = np.radians(geo["latitude"].to_numpy(float))
a = (
    np.sin((lat[:,None]-lat[None,:])/2)**2
    + np.cos(lat[:,None])*np.cos(lat[None,:])
    * np.sin((lon[:,None]-lon[None,:])/2)**2
)
km = 6371.0088 * 2 * np.arcsin(np.sqrt(np.clip(a,0,1)))
np.fill_diagonal(km, 0.0)

pair = pd.DataFrame({
    "home_nis": np.repeat(ids, len(ids)),
    "work_nis": np.tile(ids, len(ids)),
    "distance_km": km.ravel(),
})
attrs = geo.set_index("nis")
pair["same_municipality"] = pair["home_nis"].eq(pair["work_nis"])
pair["same_province"] = pair["home_nis"].map(attrs["prov_nis"]).eq(pair["work_nis"].map(attrs["prov_nis"]))
pair["same_region"] = pair["home_nis"].map(attrs["reg_nis"]).eq(pair["work_nis"].map(attrs["reg_nis"]))

# Queen-style boundary touching; diagonal is not treated as contiguity.
contig_edges = []
try:
    sindex = geo.sindex
    for i, geom in enumerate(geo.geometry):
        for j in sindex.query(geom, predicate="touches"):
            if i < j:
                contig_edges.append((geo.iloc[i]["nis"], geo.iloc[j]["nis"]))
except Exception as exc:
    warnings.warn(f"Spatial index touches failed ({exc}); using slower pairwise touches.")
    geoms = list(geo.geometry)
    for i in range(len(geoms)):
        for j in range(i+1, len(geoms)):
            if geoms[i].touches(geoms[j]):
                contig_edges.append((geo.iloc[i]["nis"], geo.iloc[j]["nis"]))

contig = pd.DataFrame(contig_edges, columns=["nis_i","nis_j"])
pair_key = set((a,b) for a,b in contig_edges) | set((b,a) for a,b in contig_edges)
pair["contiguous"] = [((a,b) in pair_key) for a,b in zip(pair.home_nis, pair.work_nis)]

write_table(pair, SUPPORT / "03_od_pair_static.parquet")
write_table(contig, SUPPORT / "08_spatial_weights" / "contiguity_edges.parquet")
print("Pair rows:", f"{len(pair):,}", "| contiguity undirected edges:", len(contig))


## 3. Population, area and density — existing verified panel or full rebuild

In [ ]:

def expected_mapping_for_year(year, target_codes):
    reverse_2019 = {new: old for old, new in RECODE_2019.items()}
    rows = []
    for target in sorted(target_codes):
        if year < 2025 and target in MERGERS_2025:
            sources, method = MERGERS_2025[target], "sum_pre2025_components"
        elif year < 2019 and target in MERGERS_2019:
            sources, method = MERGERS_2019[target], "sum_pre2019_components"
        elif year < 2019 and target in reverse_2019:
            sources, method = [reverse_2019[target]], "recode_2019"
        else:
            sources, method = [target], "direct"
        for source in sources:
            rows.append({"year":year, "source_nis":source, "nis":target, "mapping_method":method})
    out = pd.DataFrame(rows)
    require(out["source_nis"].is_unique, f"{year}: source municipality maps to multiple targets")
    return out

def _find_population_sheet(xls, year):
    if str(year) in xls.sheet_names:
        return str(year)
    cand = [s for s in xls.sheet_names if re.search(fr"(?<!\d){year}(?!\d)", str(s))]
    require(len(cand)==1, f"Cannot uniquely identify population sheet for {year}: {cand}")
    return cand[0]

def _read_population_year(path, sheet, year):
    preview = pd.read_excel(path, sheet_name=sheet, header=None, nrows=30)
    needed = {"NIS code","Woonplaats","Mannen","Vrouwen","Totaal"}
    header_rows = []
    for i, row in preview.iterrows():
        values = set(row.dropna().astype(str).str.strip())
        if needed.issubset(values):
            header_rows.append(i)
    require(len(header_rows)==1, f"{sheet}: cannot uniquely identify header")
    d = pd.read_excel(path, sheet_name=sheet, header=header_rows[0], dtype=object)
    d.columns = d.columns.astype(str).str.strip()
    d = d[["NIS code","Woonplaats","Mannen","Vrouwen","Totaal"]].rename(columns={
        "NIS code":"source_nis","Woonplaats":"source_name",
        "Mannen":"population_male","Vrouwen":"population_female","Totaal":"population"
    })
    d["source_nis"] = d["source_nis"].map(nis5)
    d = d[d["source_nis"].astype(str).str.fullmatch(r"\d{5}", na=False)].copy()
    for c in ["population","population_male","population_female"]:
        d[c] = pd.to_numeric(d[c], errors="coerce")
    is_aggregate = d["source_nis"].str.endswith("000") | d["source_nis"].isin(["20001","20002"])
    d = d.loc[~is_aggregate].copy()
    d["year"] = year
    return d

def rebuild_population_panel():
    require(POP_XLSX.exists(), f"Missing population workbook: {POP_XLSX}")
    target_codes = set(geo["nis"])
    xls = pd.ExcelFile(POP_XLSX)
    frames = []
    for y in range(YEAR_MIN-1, YEAR_MAX+1):
        sheet = _find_population_sheet(xls, y)
        frames.append(_read_population_year(POP_XLSX, sheet, y))
    rawpop = pd.concat(frames, ignore_index=True)
    cross = pd.concat([expected_mapping_for_year(y,target_codes) for y in range(YEAR_MIN-1,YEAR_MAX+1)], ignore_index=True)

    audits = []
    for y in range(YEAR_MIN-1, YEAR_MAX+1):
        exp = set(cross.loc[cross.year.eq(y),"source_nis"])
        obs = set(rawpop.loc[rawpop.year.eq(y),"source_nis"])
        audits.append({"year":y,"expected_codes":len(exp),"observed_codes":len(obs),"codes_match":exp==obs})
        require(exp==obs, f"{y}: population source codes differ from expected geography")

    mapped = rawpop.merge(cross, on=["year","source_nis"], how="left", validate="one_to_one")
    require(mapped["nis"].notna().all(), "Population has unmapped municipalities")
    pop = mapped.groupby(["nis","year"],as_index=False)[["population","population_male","population_female"]].sum(min_count=1)
    pop = pop.merge(geo[["nis","area_km2","ln_area_km2"]], on="nis", validate="many_to_one")
    pop["population_density_km2"] = pop["population"]/pop["area_km2"]
    pop["ln_population"] = np.log(pop["population"].where(pop["population"]>0))
    pop["ln_population_density"] = np.log(pop["population_density_km2"].where(pop["population_density_km2"]>0))
    pop = pop[pop.year.between(YEAR_MIN,YEAR_MAX)].copy()
    require(len(pop)==EXPECTED_N*len(YEARS), "Population panel is not balanced at 565 x 10")
    pd.DataFrame(audits).to_csv(QA_DIR/"population_rebuild_audit.csv", index=False, encoding="utf-8-sig")
    return pop

if PREFER_EXISTING_POP_PANEL and POP_PANEL_EXISTING.exists():
    population = pd.read_csv(POP_PANEL_EXISTING, dtype={"nis":str})
    population["nis"] = population["nis"].map(nis5)
    population = population[population.year.between(YEAR_MIN,YEAR_MAX)].copy()
    print("Population source: verified existing panel")
else:
    population = rebuild_population_panel()
    print("Population source: rebuilt from raw workbook")

require(len(population)==EXPECTED_N*len(YEARS), f"Population panel rows={len(population)}")
write_table(population, SUPPORT/"population_area_density_565_2015_2024.parquet")


## 4. Housing, fiscal income and administrative disposable income

In [ ]:

HOUSE_TYPE_MAP = {
    "Huizen met 2 of 3 gevels (gesloten + halfopen bebouwing)": "attached_semi",
    "Huizen met 4 of meer gevels (open bebouwing)": "detached",
    "Alle huizen met 2, 3, 4 of meer gevels (excl. appartementen)": "all_houses_excl_apartment",
    "Appartementen": "apartment",
}

def load_house_prices(path):
    frames = []
    xls = pd.ExcelFile(path)
    for year in YEARS:
        if str(year) not in xls.sheet_names:
            warnings.warn(f"Housing sheet {year} not found")
            continue
        d = pd.read_excel(path, sheet_name=str(year))
        d = d[
            (pd.to_numeric(d["CD_niveau_refnis"],errors="coerce")==5)
            & d["CD_PERIOD"].astype(str).str.upper().eq("Y")
        ].copy()
        d["year"] = pd.to_numeric(d["CD_YEAR"], errors="coerce").astype("Int64")
        d["nis"] = d["CD_REFNIS"].map(to_2025_nis)
        d["house_type"] = d["CD_TYPE_NL"].map(HOUSE_TYPE_MAP)
        d = d[d.house_type.notna()].copy()
        frames.append(d[["year","nis","house_type","MS_TOTAL_TRANSACTIONS","MS_P_50_median"]])
    long = pd.concat(frames,ignore_index=True)
    med = long.pivot_table(index=["year","nis"],columns="house_type",values="MS_P_50_median",aggfunc="first")
    med.columns = ["house_price_median_"+c for c in med.columns]
    txn = long.pivot_table(index=["year","nis"],columns="house_type",values="MS_TOTAL_TRANSACTIONS",aggfunc="first")
    txn.columns = ["house_transactions_"+c for c in txn.columns]
    out = med.join(txn,how="outer").reset_index()
    out["housing_cost_main"] = out.get("house_price_median_all_houses_excl_apartment")
    for c in ["housing_cost_main","house_price_median_apartment"]:
        if c in out:
            out["ln_"+c] = np.log(pd.to_numeric(out[c],errors="coerce").where(pd.to_numeric(out[c],errors="coerce")>0))
    return long, out

def load_tax_income(path):
    d = pd.read_excel(path, sheet_name="TF_PSNL_INC_TAX_MUNTY").copy()
    d["year"] = pd.to_numeric(d["CD_YEAR"],errors="coerce").astype("Int64")
    d = d[d.year.between(YEAR_MIN,YEAR_MAX)].copy()
    d["old_nis"] = d["CD_MUNTY_REFNIS"].map(nis5)
    d["nis"] = d["old_nis"].map(to_2025_nis)
    additive = [c for c in [
        "MS_NBR_NON_ZERO_INC","MS_NBR_ZERO_INC","MS_TOT_NET_TAXABLE_INC","MS_TOT_NET_INC",
        "MS_NBR_TOT_NET_INC","MS_REAL_ESTATE_NET_INC","MS_NBR_REAL_ESTATE_NET_INC",
        "MS_TOT_NET_MOV_ASS_INC","MS_NBR_NET_MOV_ASS_INC","MS_TOT_NET_VARIOUS_INC",
        "MS_NBR_NET_VARIOUS_INC","MS_TOT_NET_PROF_INC","MS_NBR_NET_PROF_INC",
        "MS_SEP_TAXABLE_INC","MS_NBR_SEP_TAXABLE_INC","MS_JOINT_TAXABLE_INC",
        "MS_NBR_JOINT_TAXABLE_INC","MS_TOT_DEDUCT_SPEND","MS_NBR_DEDUCT_SPEND",
        "MS_TOT_STATE_TAXES","MS_NBR_STATE_TAXES","MS_TOT_MUNICIP_TAXES",
        "MS_NBR_MUNICIP_TAXES","MS_TOT_SUBURBS_TAXES","MS_NBR_SUBURBS_TAXES",
        "MS_TOT_TAXES","MS_NBR_TOT_TAXES","MS_TOT_RESIDENTS"
    ] if c in d.columns]
    for c in additive:
        d[c] = pd.to_numeric(d[c], errors="coerce")
    out = d.groupby(["year","nis"],as_index=False)[additive].sum(min_count=1)
    if {"MS_TOT_NET_TAXABLE_INC","MS_TOT_RESIDENTS"}.issubset(out):
        out["taxable_income_per_resident"] = out["MS_TOT_NET_TAXABLE_INC"]/out["MS_TOT_RESIDENTS"].replace(0,np.nan)
        out["ln_taxable_income_per_resident"] = np.log(out["taxable_income_per_resident"].where(out["taxable_income_per_resident"]>0))
    if {"MS_TOT_NET_INC","MS_TOT_RESIDENTS"}.issubset(out):
        out["net_income_per_resident"] = out["MS_TOT_NET_INC"]/out["MS_TOT_RESIDENTS"].replace(0,np.nan)
        out["ln_net_income_per_resident"] = np.log(out["net_income_per_resident"].where(out["net_income_per_resident"]>0))
    if {"MS_TOT_TAXES","MS_TOT_RESIDENTS"}.issubset(out):
        out["taxes_per_resident"] = out["MS_TOT_TAXES"]/out["MS_TOT_RESIDENTS"].replace(0,np.nan)
    return d, out

def _wmean(v,w):
    v = pd.to_numeric(v,errors="coerce")
    w = pd.to_numeric(w,errors="coerce")
    ok = v.notna() & w.notna() & (w>0)
    return np.average(v[ok],weights=w[ok]) if ok.any() else np.nan

def load_disposable_income(path):
    d = pd.read_excel(path, sheet_name="TF_SOC_ADI_MUNTY").copy()
    d["year"] = pd.to_numeric(d["CD_YEAR"],errors="coerce").astype("Int64")
    d = d[d.year.between(YEAR_MIN,YEAR_MAX)].copy()
    d["old_nis"] = d["CD_MUNTY_REFNIS"].map(nis5)
    d["nis"] = d["old_nis"].map(to_2025_nis)
    rows = []
    for (year,nis),g in d.groupby(["year","nis"],sort=False):
        eligible = pd.to_numeric(g["MS_NBR_ELIGIBLE"],errors="coerce")
        noneligible = pd.to_numeric(g["MS_NBR_NOT_ELIGIBLE"],errors="coerce")
        es, ns = eligible.sum(min_count=1), noneligible.sum(min_count=1)
        tot = es + ns
        rows.append({
            "year":year,"nis":nis,"adi_n_components":len(g),"adi_eligible_n":es,
            "adi_not_eligible_n":ns,"adi_not_eligible_share":ns/tot if pd.notna(tot) and tot>0 else np.nan,
            "adi_q1_wapprox":_wmean(g["MS_Q1"],eligible),
            "adi_median_wapprox":_wmean(g["MS_MEDIAN"],eligible),
            "adi_q3_wapprox":_wmean(g["MS_Q3"],eligible),
            "adi_iqr_wapprox":_wmean(g["MS_INT_QUART_DIFF"],eligible),
            "adi_arop_share_wapprox":_wmean(g["MS_ADMIN_AROP"],eligible),
            "adi_ioe_hh_share_wapprox":_wmean(g["MS_PERC_IOE_HH"],eligible),
            "adi_quantile_is_approximation":len(g)>1,
        })
    out = pd.DataFrame(rows)
    out["ln_adi_median_wapprox"] = np.log(out["adi_median_wapprox"].where(out["adi_median_wapprox"]>0))
    return d, out

# Existing 136-column control panel is accepted as a verified shortcut for socioeconomic fields.
# To keep the new master interpretable, only housing/income fields are imported here; employment/network
# variables are rebuilt from the OD sources below.
if PREFER_EXISTING_SOCIO_CONTROL_PANEL and (CONTROL_PANEL_EXISTING.exists() or CONTROL_PANEL_EXISTING_CSV.exists()):
    src = CONTROL_PANEL_EXISTING if CONTROL_PANEL_EXISTING.exists() else CONTROL_PANEL_EXISTING_CSV
    old_controls = pd.read_parquet(src) if src.suffix==".parquet" else pd.read_csv(src, dtype={"nis":str})
    old_controls["nis"] = old_controls["nis"].map(nis5)
    keep = ["year","nis"] + [
        c for c in old_controls.columns
        if any(k in c.lower() for k in ["house","housing","taxable_income","net_income","taxes_per",
                                        "adi_","income_per_resident","nonzero_income"])
        and not c.endswith("_2019") and "_premean_" not in c and not c.startswith("lag1_")
    ]
    socio = old_controls[list(dict.fromkeys(keep))].copy()
    print("Socioeconomic controls: imported from verified existing panel", src)
else:
    require(HOUSE_XLSX.exists() and TAX_XLSX.exists() and ADI_XLSX.exists(), "Raw control files missing")
    _, house = load_house_prices(HOUSE_XLSX)
    _, tax = load_tax_income(TAX_XLSX)
    _, adi = load_disposable_income(ADI_XLSX)
    skeleton = pd.MultiIndex.from_product([YEARS,geo.nis],names=["year","nis"]).to_frame(index=False)
    socio = skeleton.merge(house,on=["year","nis"],how="left",validate="one_to_one")
    socio = socio.merge(tax,on=["year","nis"],how="left",validate="one_to_one")
    socio = socio.merge(adi,on=["year","nis"],how="left",validate="one_to_one")
    print("Socioeconomic controls: rebuilt from raw Statbel files")

write_table(socio, SUPPORT/"socioeconomic_controls.parquet")


## 5. Full VAR direct archive inventory and exploration helper

In [ ]:

def build_var_direct_catalog(root=VAR_DIRECT_ROOT):
    reqdir = root / "metadata" / "requests"
    if not reqdir.is_dir():
        warnings.warn(f"VAR direct metadata not found: {reqdir}")
        return pd.DataFrame(), pd.DataFrame()
    rows = []
    for p in reqdir.glob("*.json"):
        try:
            m = json.loads(p.read_text(encoding="utf-8"))
        except Exception:
            continue
        rows.append({
            "request_id":m.get("request_id",p.stem),
            "view":m.get("view"),"module":m.get("module"),"status":m.get("status"),
            "rows":m.get("rows"),"raw_file":m.get("raw_file"),
            "encoding":m.get("encoding"),"delimiter":m.get("delimiter"),
            "header_json":json.dumps(m.get("header",[]),ensure_ascii=False),
            "requested_parameters_json":json.dumps(m.get("requested_parameters",{}),ensure_ascii=False,sort_keys=True),
            "downloaded_utc":m.get("downloaded_utc"),"error":m.get("error"),
        })
    cat = pd.DataFrame(rows)
    if cat.empty:
        return cat, pd.DataFrame()
    cat["rows"] = pd.to_numeric(cat["rows"],errors="coerce")
    inv = cat.groupby(["module","view","status"],dropna=False).agg(
        requests=("request_id","count"),
        total_returned_rows=("rows","sum"),
        nonempty_requests=("rows",lambda s:int((s.fillna(0)>0).sum()))
    ).reset_index()
    return cat, inv

def load_var_direct_view(view_name, max_files=None, parameter_equals=None):
    if "var_catalog" not in globals() or var_catalog.empty:
        raise ValueError("Run the VAR catalog cell first.")
    q = var_catalog[(var_catalog.view==view_name)&(var_catalog.status=="received")].copy()
    if parameter_equals:
        def ok(txt):
            d = json.loads(txt)
            return all(str(d.get(k))==str(v) for k,v in parameter_equals.items())
        q = q[q.requested_parameters_json.map(ok)]
    if max_files is not None:
        q = q.head(max_files)
    pieces = []
    for r in q.itertuples(index=False):
        raw = VAR_DIRECT_ROOT / str(r.raw_file)
        if not raw.exists():
            continue
        params = json.loads(r.requested_parameters_json)
        enc = r.encoding or "utf-8-sig"
        sep = r.delimiter or ","
        with gzip.open(raw,"rt",encoding=enc,errors="replace",newline="") as f:
            d = pd.read_csv(f,sep=sep,low_memory=False)
        d["_request_id"] = r.request_id
        for k,v in params.items():
            d["req__"+ascii_slug(k)] = v
        pieces.append(d)
    return pd.concat(pieces,ignore_index=True) if pieces else pd.DataFrame()

if BUILD_FULL_VAR_CATALOG:
    var_catalog, var_inventory = build_var_direct_catalog()
    if not var_catalog.empty:
        write_table(var_catalog, SUPPORT/"var_direct_request_catalog.parquet")
        var_inventory.to_csv(QA_DIR/"var_direct_view_inventory.csv",index=False,encoding="utf-8-sig")
        print(var_inventory.sort_values(["module","view"]).to_string(index=False))
    else:
        print("No direct VAR catalog available.")
else:
    var_catalog, var_inventory = pd.DataFrame(), pd.DataFrame()

# Deliberately off by default: concatenating all 7,118 source responses can be very large.
if MATERIALIZE_FULL_VAR_VIEWS and not var_inventory.empty:
    target = SUPPORT/"var_direct_views"
    target.mkdir(exist_ok=True)
    for view in sorted(var_catalog.loc[var_catalog.status.eq("received"),"view"].dropna().unique()):
        if "tabel" not in norm(view) and view not in ("Pendel In - Tabel","Pendel Uit - Tabel"):
            continue
        print("Materialising:", view)
        d = load_var_direct_view(view)
        if len(d):
            d.to_parquet(target/(ascii_slug(view)+".parquet"),index=False)


## 6. Load processed OD layers and construct the observed domestic network

In [ ]:

NODE_TO_NIS = geo.set_index("node_id")["nis"].to_dict()
NIS_SET = set(geo["nis"])

def _header(path):
    return pd.read_csv(path,nrows=0,encoding="utf-8-sig").columns.tolist()

def load_od(path, dimension=None, age_scope="20-64"):
    require(Path(path).exists(), f"Missing OD file: {path}")
    cols = _header(path)
    use = ["year","home_id","work_id","workers"]
    dimcol = dimension
    if dimcol and dimcol in cols:
        use.append(dimcol)
    if "age_group" in cols and "age_group" not in use:
        use.append("age_group")
    d = pd.read_csv(path,usecols=use,encoding="utf-8-sig",low_memory=False)
    d["year"] = pd.to_numeric(d["year"],errors="coerce")
    d = d[d.year.isin(YEARS)].copy()
    d["year"] = d["year"].astype(int)
    d["workers"] = pd.to_numeric(d["workers"],errors="coerce")
    require((d["workers"].dropna()>=0).all(), f"Negative OD flow in {path}")
    if dimension != "age" and "age_group" in d.columns and age_scope:
        labels = d["age_group"].astype(str).str.replace("–","-",regex=False)
        mask = labels.str.contains(age_scope,regex=False,na=False)
        if mask.any():
            d = d[mask].copy()
    d["home_id"] = d["home_id"].astype(str).str.strip()
    d["work_id"] = d["work_id"].astype(str).str.strip()
    d["home_nis"] = d["home_id"].map(NODE_TO_NIS)
    d["work_nis"] = d["work_id"].map(NODE_TO_NIS)
    return d

od_main_raw = load_od(OD_FILES["total"])
od_domestic = od_main_raw[od_main_raw.home_nis.notna() & od_main_raw.work_nis.notna()].copy()
od_domestic = od_domestic.groupby(["year","home_nis","work_nis"],as_index=False)["workers"].sum(min_count=1)
od_domestic = od_domestic.merge(pair[["home_nis","work_nis","distance_km","same_municipality",
                                      "same_province","same_region","contiguous"]],
                                on=["home_nis","work_nis"],how="left",validate="many_to_one")
od_domestic["observed_in_source"] = True
od_domestic["pair_id"] = od_domestic["home_nis"]+"__"+od_domestic["work_nis"]

# External-node summaries are retained separately rather than discarded from resident/workplace totals.
external_summary = pd.concat([
    od_main_raw[od_main_raw.home_nis.notna() & od_main_raw.work_nis.isna()]
      .groupby(["year","home_nis"],as_index=False)["workers"].sum(min_count=1)
      .rename(columns={"workers":"resident_workers_external"}).assign(side="residence"),
    od_main_raw[od_main_raw.work_nis.notna() & od_main_raw.home_nis.isna()]
      .groupby(["year","work_nis"],as_index=False)["workers"].sum(min_count=1)
      .rename(columns={"workers":"workplace_workers_external_origin"}).assign(side="workplace"),
],ignore_index=True,sort=False)

write_table(od_domestic, SUPPORT/"od_observed_domestic.parquet")
write_table(external_summary, SUPPORT/"od_external_node_summary.parquet")
print("Observed domestic OD rows:", f"{len(od_domestic):,}")


## 7. Municipality employment scale and residence/workplace composition

In [ ]:

# Basic employment quantities use all mapped resident/workplace records, including one-sided external links.
resident_workers = (
    od_main_raw[od_main_raw.home_nis.notna()]
    .groupby(["year","home_nis"],as_index=False)["workers"].sum(min_count=1)
    .rename(columns={"home_nis":"nis","workers":"resident_workers"})
)
workplace_jobs = (
    od_main_raw[od_main_raw.work_nis.notna()]
    .groupby(["year","work_nis"],as_index=False)["workers"].sum(min_count=1)
    .rename(columns={"work_nis":"nis","workers":"workplace_jobs"})
)
local_workers = (
    od_domestic[od_domestic.same_municipality]
    .groupby(["year","home_nis"],as_index=False)["workers"].sum(min_count=1)
    .rename(columns={"home_nis":"nis","workers":"local_workers"})
)
employment = resident_workers.merge(workplace_jobs,on=["year","nis"],how="outer",validate="one_to_one")
employment = employment.merge(local_workers,on=["year","nis"],how="left",validate="one_to_one")
employment["local_workers"] = employment["local_workers"].fillna(0)
employment = employment.merge(geo[["nis","area_km2"]],on="nis",validate="many_to_one")
employment["employment_density_km2"] = employment["workplace_jobs"]/employment["area_km2"]
employment["jobs_to_resident_workers"] = employment["workplace_jobs"]/employment["resident_workers"].replace(0,np.nan)
employment["resident_local_job_share"] = employment["local_workers"]/employment["resident_workers"].replace(0,np.nan)
employment["workplace_local_worker_share"] = employment["local_workers"]/employment["workplace_jobs"].replace(0,np.nan)
employment["ln_workplace_jobs"] = np.log(employment["workplace_jobs"].where(employment["workplace_jobs"]>0))
employment["ln_resident_workers"] = np.log(employment["resident_workers"].where(employment["resident_workers"]>0))
employment["ln_employment_density_km2"] = np.log(employment["employment_density_km2"].where(employment["employment_density_km2"]>0))
write_table(employment, SUPPORT/"employment_scale_municipality_year.parquet")

DIM_CANON = {
    "education":{
        "hooggeschoold":"high","middengeschoold":"middle","kortgeschoold":"low",
        "onbekend":"unknown","totaal":"total","total":"total"
    },
    "sex":{
        "mannen":"male","man":"male","male":"male","vrouwen":"female","vrouw":"female","female":"female",
        "onbekend":"unknown","mannen en vrouwen":"total","totaal":"total","total":"total"
    },
    "employment_status":{
        "loontrekkend":"employee","loontrekkenden":"employee","employee":"employee",
        "zelfstandig":"self_employed","zelfstandigen":"self_employed",
        "werkend":"working","werkenden":"working","totaal":"total","total":"total"
    }
}

def canonical_group(value, dim):
    s = norm(value)
    return DIM_CANON.get(dim,{}).get(s, ascii_slug(value) if s else "missing")

def dimension_node_shares(path, dimcol, dimname, age_scope="20-64"):
    if not Path(path).exists():
        return pd.DataFrame(), pd.DataFrame()
    d = load_od(path, dimension=dimcol, age_scope=age_scope)
    require(dimcol in d.columns, f"{path}: {dimcol} missing")
    d["category"] = d[dimcol].map(lambda x: canonical_group(x,dimname))
    outputs, longs = [], []
    for side,nodecol,prefix in [("residence","home_nis","resident"),("workplace","work_nis","workplace")]:
        x = d[d[nodecol].notna()].copy()
        g = x.groupby(["year",nodecol,"category"],as_index=False)["workers"].sum(min_count=1)
        g = g.rename(columns={nodecol:"nis"})
        longs.append(g.assign(side=side))
        totals = g[g.category.eq("total")][["year","nis","workers"]].rename(columns={"workers":"published_total"})
        detail = g[~g.category.eq("total")].copy()
        wide = detail.pivot_table(index=["year","nis"],columns="category",values="workers",aggfunc="sum").reset_index()
        wide.columns = ["year","nis"] + [prefix+"_"+dimname+"_"+c+"_workers" for c in wide.columns[2:]]
        wide = wide.merge(totals,on=["year","nis"],how="left",validate="one_to_one")
        detail_cols = [c for c in wide.columns if c.endswith("_workers")]
        for c in detail_cols:
            wide[c.replace("_workers","_share")] = wide[c]/wide["published_total"].replace(0,np.nan)
        wide = wide.rename(columns={"published_total":prefix+"_"+dimname+"_published_total"})
        outputs.append(wide)
    out = outputs[0].merge(outputs[1],on=["year","nis"],how="outer",validate="one_to_one")
    return pd.concat(longs,ignore_index=True), out

education_long, education_wide = dimension_node_shares(OD_FILES["education"],"education","education")
sex_long, sex_wide = dimension_node_shares(OD_FILES["sex"],"sex","sex")
status_long, status_wide = dimension_node_shares(OD_FILES["employment_status"],"employment_status","employment_status")

for name, table in [("education",education_long),("sex",sex_long),("employment_status",status_long)]:
    if len(table):
        write_table(table,SUPPORT/(name+"_node_long.parquet"))

# Age is preserved in long form because published age intervals overlap.
if OD_FILES["age"].exists():
    age_raw = load_od(OD_FILES["age"],dimension="age_group",age_scope=None)
    age_raw["age_category"] = age_raw["age_group"].astype(str).map(ascii_slug)
    age_parts = []
    for side,nodecol in [("residence","home_nis"),("workplace","work_nis")]:
        q = age_raw[age_raw[nodecol].notna()].groupby(["year",nodecol,"age_category"],as_index=False)["workers"].sum(min_count=1)
        q = q.rename(columns={nodecol:"nis"}).assign(side=side)
        age_parts.append(q)
    age_long = pd.concat(age_parts,ignore_index=True)
    write_table(age_long,SUPPORT/"age_node_long.parquet")
else:
    age_long = pd.DataFrame()


## 8. Sector structure and WSE42 long tables

In [ ]:

WSE_TO_NACE = {
    "p1":"A;B",
    **{f"s{i}":"C" for i in range(1,14)},
    "s14":"D;E","s15":"E","s16":"F",
    "t1":"C;S","t2":"G","t3":"G","t4":"G","t5":"H","t6":"H","t7":"H",
    "t8":"I;N","t9":"J","t10":"J","t11":"J","t12":"K","t13":"M","t14":"N","t15":"N",
    "t16":"L;N","t17":"M;S;T",
    "q1":"R","q2":"O","q3":"O;U","q4":"O","q5":"P","q6":"Q","q7":"Q","q8":"S"
}
WSE_CORE_SINGLE_NACE = {k:v for k,v in WSE_TO_NACE.items() if ";" not in v}

sector_raw = load_od(OD_FILES["sector"],dimension="sector",age_scope="20-64")
sector_raw["sector_code"] = sector_raw["sector"].astype(str).str.extract(r"^([pqst]\d+)\b",flags=re.I)[0].str.lower()
sector_raw = sector_raw[sector_raw.sector_code.isin(WSE_TO_NACE)].copy()

sector_parts = []
for side,nodecol in [("residence","home_nis"),("workplace","work_nis")]:
    q = (
        sector_raw[sector_raw[nodecol].notna()]
        .groupby(["year",nodecol,"sector_code"],as_index=False)["workers"].sum(min_count=1)
        .rename(columns={nodecol:"nis"})
        .assign(side=side)
    )
    sector_parts.append(q)
sector_long = pd.concat(sector_parts,ignore_index=True)

main_side_total = pd.concat([
    resident_workers.rename(columns={"resident_workers":"published_total"}).assign(side="residence"),
    workplace_jobs.rename(columns={"workplace_jobs":"published_total"}).assign(side="workplace"),
],ignore_index=True)
sector_long = sector_long.merge(main_side_total,on=["side","year","nis"],how="left",validate="many_to_one")
sector_long["share_of_main_total"] = sector_long["workers"]/sector_long["published_total"].replace(0,np.nan)
write_table(sector_long,SUPPORT/"06_sector_panel_residence_workplace.parquet")
write_table(sector_long[sector_long.side.eq("residence")].drop(columns="side"),
            SUPPORT/"06_sector_panel_residence.parquet")
write_table(sector_long[sector_long.side.eq("workplace")].drop(columns="side"),
            SUPPORT/"07_sector_panel_workplace.parquet")

# Compact municipality-year sector summaries for the master panel.
sector_summary_rows = []
for (side,year,nis),g in sector_long.groupby(["side","year","nis"],observed=True):
    total = g["published_total"].dropna()
    total = float(total.iloc[0]) if len(total) else np.nan
    known = g["workers"].sum(min_count=1)
    w = g["workers"].to_numpy(float)
    wf = w[np.isfinite(w) & (w >= 0)]
    p = wf/wf.sum() if len(wf) and wf.sum()>0 else np.array([])
    row = {
        "side":side,"year":year,"nis":nis,"sector_known_workers":known,
        "sector_coverage_vs_main":known/total if pd.notna(total) and total>0 else np.nan,
        "sector_hhi_known":float(np.sum(p*p)) if len(p) else np.nan,
        "sector_entropy_known":float(-np.sum(p[p>0]*np.log(p[p>0]))) if len(p) else np.nan,
    }
    for macro,prefix in [("primary","p"),("secondary","s"),("tertiary","t"),("quaternary","q")]:
        m = g.loc[g.sector_code.str.startswith(prefix),"workers"].sum(min_count=1)
        row[macro+"_share_main"] = m/total if pd.notna(total) and total>0 else np.nan
    sector_summary_rows.append(row)
sector_summary = pd.DataFrame(sector_summary_rows)

res_sector_summary = sector_summary[sector_summary.side.eq("residence")].drop(columns="side").rename(
    columns={c:"resident_"+c for c in sector_summary.columns if c not in ["side","year","nis"]})
work_sector_summary = sector_summary[sector_summary.side.eq("workplace")].drop(columns="side").rename(
    columns={c:"workplace_"+c for c in sector_summary.columns if c not in ["side","year","nis"]})
sector_summary_wide = res_sector_summary.merge(work_sector_summary,on=["year","nis"],how="outer",validate="one_to_one")
write_table(sector_summary_wide,SUPPORT/"sector_structure_summary.parquet")


## 9. Network outcomes and node-level topology — residence and workplace sides

In [ ]:

def one_node_network(g, node_col, partner_col):
    w = g["workers"].to_numpy(float)
    d = g["distance_km"].to_numpy(float)
    node = str(g[node_col].iloc[0])
    partner = g[partner_col].astype(str).to_numpy()
    ext = partner != node
    positive = np.isfinite(w) & (w>0)
    total = np.nansum(w)
    cross = np.nansum(w[ext])
    local = np.nansum(w[~ext])
    shares = w[positive]/np.nansum(w[positive]) if positive.any() else np.array([])
    hhi = np.sum(shares**2) if len(shares) else np.nan
    entropy = -np.sum(shares*np.log(shares)) if len(shares) else np.nan
    sorted_shares = np.sort(shares)[::-1] if len(shares) else np.array([])
    q50,q75,q90 = weighted_quantile(d,w,(.5,.75,.9))
    de = d[ext]; we = w[ext]
    e50,e75,e90 = weighted_quantile(de,we,(.5,.75,.9))
    return pd.Series({
        "workers":total,
        "local_workers":local,
        "cross_workers":cross,
        "local_share":local/total if total>0 else np.nan,
        "cross_share":cross/total if total>0 else np.nan,
        "mean_km":np.nansum(w*d)/total if total>0 else np.nan,
        "external_mean_km":np.nansum(we*de)/cross if cross>0 else np.nan,
        "long30_share":np.nansum(w[d>30])/total if total>0 else np.nan,
        "long50_share":np.nansum(w[d>50])/total if total>0 else np.nan,
        "external_long30_share":np.nansum(we[de>30])/cross if cross>0 else np.nan,
        "external_long50_share":np.nansum(we[de>50])/cross if cross>0 else np.nan,
        "p50_km":q50,"p75_km":q75,"p90_km":q90,
        "external_p50_km":e50,"external_p75_km":e75,"external_p90_km":e90,
        "n_published_links":len(g),
        "n_positive_links":int(positive.sum()),
        "external_positive_partners":int((positive & ext).sum()),
        "flow_hhi":hhi,
        "flow_entropy":entropy,
        "effective_partners":float(np.exp(entropy)) if pd.notna(entropy) else np.nan,
        "top1_flow_share":sorted_shares[0] if len(sorted_shares) else np.nan,
        "top3_flow_share":sorted_shares[:3].sum() if len(sorted_shares) else np.nan,
    })

def build_node_network_metrics(od):
    res = (
        od.groupby(["year","home_nis"],observed=True,group_keys=False)
          .apply(lambda g: one_node_network(g,"home_nis","work_nis"))
          .reset_index().rename(columns={"home_nis":"nis"})
    )
    work = (
        od.groupby(["year","work_nis"],observed=True,group_keys=False)
          .apply(lambda g: one_node_network(g,"work_nis","home_nis"))
          .reset_index().rename(columns={"work_nis":"nis"})
    )
    res = res.rename(columns={c:"resnet_"+c for c in res.columns if c not in ["year","nis"]})
    work = work.rename(columns={c:"worknet_"+c for c in work.columns if c not in ["year","nis"]})
    return res, work

resnet, worknet = build_node_network_metrics(od_domestic)
write_table(resnet,SUPPORT/"04_network_metrics_residence.parquet")
write_table(worknet,SUPPORT/"04_network_metrics_workplace.parquet")
print("Residence network panel:",resnet.shape,"Workplace network panel:",worknet.shape)


## 10. Accessibility from the same 565-municipality distance system

In [ ]:

id_order = geo["nis"].tolist()
D = pair.pivot(index="home_nis",columns="work_nis",values="distance_km").reindex(index=id_order,columns=id_order).to_numpy(float)
W_EXP50 = np.exp(-D/50.0)
W_EXP100 = np.exp(-D/100.0)
W_INV = 1.0/(1.0+D)
np.fill_diagonal(W_EXP50,1.0)
np.fill_diagonal(W_EXP100,1.0)
np.fill_diagonal(W_INV,1.0)

def accessibility_from_panel(panel, value_col, prefix):
    rows = []
    for y in YEARS:
        s = panel[panel.year.eq(y)].set_index("nis")[value_col].reindex(id_order)
        x = s.to_numpy(float)
        valid = np.isfinite(x)
        if not valid.any():
            continue
        for name,W in [("exp50",W_EXP50),("exp100",W_EXP100),("inv",W_INV)]:
            num = W[:,valid] @ x[valid]
            # Not row-normalised: this is Hansen-style potential accessibility.
            if name=="exp50":
                vals50 = num
            elif name=="exp100":
                vals100 = num
            else:
                valsinv = num
        rows.append(pd.DataFrame({
            "year":y,"nis":id_order,
            prefix+"_access_exp50":vals50,
            prefix+"_access_exp100":vals100,
            prefix+"_access_inv":valsinv,
        }))
    out = pd.concat(rows,ignore_index=True)
    for c in [prefix+"_access_exp50",prefix+"_access_exp100",prefix+"_access_inv"]:
        out["ln_"+c] = np.log(out[c].where(out[c]>0))
    return out

job_access = accessibility_from_panel(employment[["year","nis","workplace_jobs"]],"workplace_jobs","job")
pop_access = accessibility_from_panel(population[["year","nis","population"]],"population","population")
accessibility = job_access.merge(pop_access,on=["year","nis"],how="outer",validate="one_to_one")
write_table(accessibility,SUPPORT/"accessibility_municipality_year.parquet")


## 11. Telework/WFH exposures: current, frozen-share, Bartik, and 2018/2019 baseline variants

In [ ]:

def load_wfh_rates():
    if WFH_RATE_CSV.exists():
        r = pd.read_csv(WFH_RATE_CSV,encoding="utf-8-sig")
        need = {"year","nace_section","any_wfh_rate"}
        require(need.issubset(r.columns), f"{WFH_RATE_CSV} missing {need-set(r.columns)}")
        r["year"] = pd.to_numeric(r["year"],errors="coerce").astype("Int64")
        r["nace_section"] = r["nace_section"].astype(str).str.strip().str.upper()
        r["any_wfh_rate"] = pd.to_numeric(r["any_wfh_rate"],errors="coerce")
        r = r[r.nace_section.str.fullmatch(r"[A-U]",na=False)].copy()
        require(((r.any_wfh_rate.dropna()>=0)&(r.any_wfh_rate.dropna()<=1)).all(),"WFH rate outside [0,1]")
        return r
    warnings.warn(
        "Verified NACE-year WFH rate CSV not found. Exposure construction is skipped. "
        "Run the earlier telework builder or point WFH_RATE_CSV to its 01_nace_year_wfh_rates_2010_2025.csv output."
    )
    return pd.DataFrame()

rates = load_wfh_rates()

def build_current_sector_exposure(sector_long, rates):
    if rates.empty:
        return pd.DataFrame()
    mapdf = pd.DataFrame([{"sector_code":k,"nace_section":v} for k,v in WSE_CORE_SINGLE_NACE.items()])
    x = sector_long.merge(mapdf,on="sector_code",how="left",validate="many_to_one")
    x = x.merge(rates[["year","nace_section","any_wfh_rate"]],on=["year","nace_section"],how="left",validate="many_to_one")
    x["supported_worker_rate"] = x["workers"]*x["any_wfh_rate"]
    rows = []
    for (side,year,nis),g in x.groupby(["side","year","nis"],observed=True):
        total = g["published_total"].dropna()
        total = float(total.iloc[0]) if len(total) else np.nan
        core = g[g.nace_section.notna() & g.any_wfh_rate.notna()].copy()
        core_workers = core["workers"].sum(min_count=1)
        value = core["supported_worker_rate"].sum(min_count=1)/core_workers if pd.notna(core_workers) and core_workers>0 else np.nan
        rows.append({
            "side":side,"year":year,"nis":nis,
            "tw_current":value,
            "tw_core_workers":core_workers,
            "tw_core_coverage_total":core_workers/total if pd.notna(total) and total>0 else np.nan,
        })
    return pd.DataFrame(rows)

def _nace_worker_matrix(sector_long, side):
    mapdf = pd.DataFrame([{"sector_code":k,"nace_section":v} for k,v in WSE_CORE_SINGLE_NACE.items()])
    x = sector_long[sector_long.side.eq(side)].merge(mapdf,on="sector_code",how="inner",validate="many_to_one")
    return x.groupby(["nis","year","nace_section"],as_index=False)["workers"].sum(min_count=1)

def build_frozen_and_bartik(sector_long, rates, side, share_year=2019):
    x = _nace_worker_matrix(sector_long,side)
    if x.empty or rates.empty:
        return pd.DataFrame()
    base = x[x.year.eq(share_year)].copy()
    base = base.merge(rates[rates.year.eq(share_year)][["nace_section","any_wfh_rate"]]
                      .rename(columns={"any_wfh_rate":"rate_base"}),on="nace_section",how="left",validate="many_to_one")
    base = base[base.rate_base.notna()].copy()
    den = base.groupby("nis")["workers"].transform("sum")
    base["w_share"] = base["workers"]/den.replace(0,np.nan)
    base_exp = (base.assign(z=base.w_share*base.rate_base).groupby("nis",as_index=False)["z"].sum()
                .rename(columns={"z":f"tw_{prefix}_s{share_year}_r{share_year}"}))
    rows = []
    for y in sorted(set(YEARS)&set(pd.to_numeric(rates.year,errors="coerce").dropna().astype(int))):
        rr = rates[rates.year.eq(y)][["nace_section","any_wfh_rate"]]
        q = base.merge(rr,on="nace_section",how="left",validate="many_to_one")
        q["weighted_rate"] = q["w_share"]*q["any_wfh_rate"]
        z = q.groupby("nis").agg(
            frozen=("weighted_rate","sum"),
            positive_weight=("w_share","sum"),
            missing_rate_on_positive_weight=("any_wfh_rate",lambda s:int(s.isna().sum()))
        ).reset_index()
        z["year"] = y
        z[f"tw_{prefix}_frozen{share_year}"] = z["frozen"].where(z.missing_rate_on_positive_weight.eq(0))
        rows.append(z[["year","nis",f"tw_{prefix}_frozen{share_year}","positive_weight","missing_rate_on_positive_weight"]])
    out = pd.concat(rows,ignore_index=True)
    out = out.merge(base_exp,on="nis",how="left",validate="many_to_one")
    basecol = f"tw_{prefix}_s{share_year}_r{share_year}"
    out[f"bartik_{prefix}_{share_year}"] = out[f"tw_{prefix}_frozen{share_year}"]-out[basecol]
    return out

def baseline_2x2(sector_long,rates,side_value,prefix):
    x = _nace_worker_matrix(sector_long,side_value)
    if x.empty or rates.empty:
        return pd.DataFrame()
    p = x[x.year.isin([2018,2019])].pivot_table(index=["nis","nace_section"],columns="year",values="workers",aggfunc="sum")
    p = p.reset_index()
    for y in (2018,2019):
        if y not in p:
            p[y] = np.nan
    rr = rates[rates.year.isin([2018,2019])].pivot(index="nace_section",columns="year",values="any_wfh_rate")
    p["r2018"] = p.nace_section.map(rr[2018] if 2018 in rr else pd.Series(dtype=float))
    p["r2019"] = p.nace_section.map(rr[2019] if 2019 in rr else pd.Series(dtype=float))
    p["support"] = p[2018].notna() & p[2019].notna() & p.r2018.notna() & p.r2019.notna()
    p = p[p.support].copy()
    rows = []
    for nis,g in p.groupby("nis"):
        row={"nis":nis,"shared_core_cells":len(g)}
        for sy in (2018,2019):
            den = g[sy].sum()
            if den<=0:
                continue
            w = g[sy]/den
            for ry in (2018,2019):
                row[f"tw_{prefix}_s{sy}_r{ry}"] = float(np.sum(w*g[f"r{ry}"]))
        rows.append(row)
    return pd.DataFrame(rows)

if not rates.empty:
    current = build_current_sector_exposure(sector_long,rates)
    current_res = current[current.side.eq("residence")].drop(columns="side").rename(
        columns={"tw_current":"tw_res_current","tw_core_workers":"tw_res_core_workers",
                 "tw_core_coverage_total":"tw_res_core_coverage_total"})
    current_work = current[current.side.eq("workplace")].drop(columns="side").rename(
        columns={"tw_current":"tw_work_current","tw_core_workers":"tw_work_core_workers",
                 "tw_core_coverage_total":"tw_work_core_coverage_total"})
    exposure_panel = current_res.merge(current_work,on=["year","nis"],how="outer",validate="one_to_one")
    exposure_panel = exposure_panel.merge(build_frozen_and_bartik(sector_long,rates,"residence","res",2019),on=["year","nis"],how="outer",validate="one_to_one")
    exposure_panel = exposure_panel.merge(build_frozen_and_bartik(sector_long,rates,"workplace","work",2019),on=["year","nis"],how="outer",validate="one_to_one")
    base_res = baseline_2x2(sector_long,rates,"residence","res")
    base_work = baseline_2x2(sector_long,rates,"workplace","work")
    base = base_res.merge(base_work,on="nis",how="outer",validate="one_to_one")
    exposure_panel = exposure_panel.merge(base,on="nis",how="left",validate="many_to_one")
else:
    exposure_panel = pd.MultiIndex.from_product([YEARS,geo.nis],names=["year","nis"]).to_frame(index=False)

# Optional preferred workplace baseline from the verified formal workplace-margin component.
official_work_baseline = pd.DataFrame()
if not rates.empty and WFH_WORKPLACE_MARGIN_CSV.exists():
    m = pd.read_csv(WFH_WORKPLACE_MARGIN_CSV,encoding="utf-8-sig")
    m["year"] = pd.to_numeric(m["year"],errors="coerce").astype("Int64")
    if "nis" not in m.columns and "node_id" in m.columns:
        m["nis"] = m["node_id"].map(NODE_TO_NIS)
    m["nis"] = m["nis"].map(nis5)
    m["wse_code"] = m["wse_code"].astype(str).str.lower().str.strip()
    m["jobs"] = pd.to_numeric(m["jobs"],errors="coerce")
    mm = m[m.wse_code.isin(WSE_CORE_SINGLE_NACE)].copy()
    mm["nace_section"] = mm["wse_code"].map(WSE_CORE_SINGLE_NACE)
    mm = mm.groupby(["nis","year","nace_section"],as_index=False)["jobs"].sum(min_count=1)
    pseudo = mm.rename(columns={"jobs":"workers"}).assign(side="workplace",published_total=np.nan,sector_code="x")
    # Reuse a compact 2x2 implementation directly on NACE rows.
    p = mm[mm.year.isin([2018,2019])].pivot_table(index=["nis","nace_section"],columns="year",values="jobs",aggfunc="sum").reset_index()
    for y in (2018,2019):
        if y not in p: p[y]=np.nan
    rr = rates[rates.year.isin([2018,2019])].pivot(index="nace_section",columns="year",values="any_wfh_rate")
    p["r2018"]=p.nace_section.map(rr[2018]); p["r2019"]=p.nace_section.map(rr[2019])
    p=p[p[2018].notna()&p[2019].notna()&p.r2018.notna()&p.r2019.notna()]
    rec=[]
    for nis,g in p.groupby("nis"):
        row={"nis":nis}
        for sy in (2018,2019):
            den=g[sy].sum()
            if den>0:
                w=g[sy]/den
                for ry in (2018,2019):
                    row[f"tw_work_official_s{sy}_r{ry}"]=float(np.sum(w*g[f"r{ry}"]))
        rec.append(row)
    official_work_baseline=pd.DataFrame(rec)
    exposure_panel=exposure_panel.merge(official_work_baseline,on="nis",how="left",validate="many_to_one")

if len(exposure_panel):
    write_table(exposure_panel,SUPPORT/"05_telework_exposure_municipality_year.parquet")
    if len(official_work_baseline):
        write_table(official_work_baseline,SUPPORT/"telework_official_workplace_baseline.parquet")


## 12. Pre-pandemic commuting-network exposure and spatial spillover exposure

In [ ]:

# Pre-pandemic domestic OD weights. Both any-published and stable-2017-2019 versions are retained.
pre = od_domestic[od_domestic.year.isin([2017,2018,2019])].copy()
pre_pair = pre.groupby(["home_nis","work_nis"]).agg(
    pre_flow_sum=("workers","sum"),
    n_pre_years=("year","nunique")
).reset_index()

def add_weights(df, side):
    node = "home_nis" if side=="out" else "work_nis"
    q = df.copy()
    den = q.groupby(node)["pre_flow_sum"].transform("sum")
    q[f"w_pre_{side}_any"] = q["pre_flow_sum"]/den.replace(0,np.nan)
    stable = q.n_pre_years.eq(3)
    stabden = q["pre_flow_sum"].where(stable,0).groupby(q[node]).transform("sum")
    q[f"w_pre_{side}_stable"] = np.where(stable,q["pre_flow_sum"]/stabden.replace(0,np.nan),0.0)
    return q

pre_weights = add_weights(add_weights(pre_pair,"out"),"in")
write_table(pre_weights,SUPPORT/"09_network_weights_pre2019"/"commuting_weights_2017_2019.parquet")

def network_weighted_exposure(exposure_panel, var, direction="out", stable=False):
    if var not in exposure_panel.columns:
        return pd.DataFrame()
    wcol = f"w_pre_{direction}_{'stable' if stable else 'any'}"
    if direction=="out":
        node, other = "home_nis","work_nis"
        prefix = "out"
    else:
        node, other = "work_nis","home_nis"
        prefix = "in"
    w = pre_weights[[node,other,wcol]].copy()
    rows=[]
    for y in YEARS:
        e = exposure_panel[exposure_panel.year.eq(y)][["nis",var]].rename(columns={"nis":other})
        q = w.merge(e,on=other,how="left",validate="many_to_one")
        q["valid_w"] = q[wcol].where(q[var].notna(),0)
        q["wx"] = q[wcol]*q[var]
        a = q.groupby(node).agg(wx=("wx","sum"),coverage=("valid_w","sum")).reset_index()
        a["value"] = a["wx"]/a["coverage"].replace(0,np.nan)
        a["year"]=y
        a=a.rename(columns={node:"nis","value":f"network_{prefix}_{var}_{'stable' if stable else 'any'}",
                            "coverage":f"network_{prefix}_{var}_{'stable' if stable else 'any'}_coverage"})
        rows.append(a[["year","nis",f"network_{prefix}_{var}_{'stable' if stable else 'any'}",
                       f"network_{prefix}_{var}_{'stable' if stable else 'any'}_coverage"]])
    return pd.concat(rows,ignore_index=True)

network_exposure = pd.MultiIndex.from_product([YEARS,geo.nis],names=["year","nis"]).to_frame(index=False)
for var,direction in [
    ("tw_work_current","out"),
    ("bartik_work_2019","out"),
    ("tw_res_current","in"),
    ("bartik_res_2019","in"),
]:
    for stable in (False,True):
        z=network_weighted_exposure(exposure_panel,var,direction,stable)
        if len(z):
            network_exposure=merge_unique(network_exposure,z,label=f"network {var}")
write_table(network_exposure,SUPPORT/"network_weighted_telework_exposure.parquet")

# Spatial-weight matrices: exclude self, row-normalise; missing exposure values are renormalised at evaluation time.
N=len(id_order)
CONT=np.zeros((N,N),float)
idx={n:i for i,n in enumerate(id_order)}
for a,b in contig_edges:
    CONT[idx[a],idx[b]]=1; CONT[idx[b],idx[a]]=1
D_NOSELF=D.copy()
np.fill_diagonal(D_NOSELF,np.nan)
DIST50=np.exp(-np.nan_to_num(D_NOSELF,nan=np.inf)/50.0); np.fill_diagonal(DIST50,0)
INVD=np.divide(1.0,D_NOSELF,out=np.zeros_like(D_NOSELF),where=np.isfinite(D_NOSELF)&(D_NOSELF>0))
np.fill_diagonal(INVD,0)

def spatial_lag_for_var(panel,var,W,scheme):
    if var not in panel.columns:
        return pd.DataFrame()
    rows=[]
    for y in YEARS:
        x=panel[panel.year.eq(y)].set_index("nis")[var].reindex(id_order).to_numpy(float)
        valid=np.isfinite(x)
        denom=W[:,valid].sum(axis=1)
        val=np.divide(W[:,valid]@x[valid],denom,out=np.full(N,np.nan),where=denom>0)
        rows.append(pd.DataFrame({"year":y,"nis":id_order,f"spatial_{scheme}_{var}":val}))
    return pd.concat(rows,ignore_index=True)

spatial_exposure=pd.MultiIndex.from_product([YEARS,geo.nis],names=["year","nis"]).to_frame(index=False)
for var in ["tw_work_current","tw_res_current","bartik_work_2019","bartik_res_2019"]:
    for scheme,W in [("contig",CONT),("dist50",DIST50),("invdist",INVD)]:
        z=spatial_lag_for_var(exposure_panel,var,W,scheme)
        if len(z):
            spatial_exposure=merge_unique(spatial_exposure,z,label=f"spatial {var}")
write_table(spatial_exposure,SUPPORT/"spatial_telework_exposure.parquet")


## 13. Build the municipality × year master panel

In [ ]:

skeleton = pd.MultiIndex.from_product([YEARS,id_order],names=["year","nis"]).to_frame(index=False)
geo_attrs = geo.drop(columns="geometry").copy()
geo_attrs = geo_attrs[[c for c in [
    "nis","node_id","var_name","name_nl","name_fr","arr_nis","prov_nis","reg_nis",
    "boundary_reference","longitude","latitude","area_km2","ln_area_km2"
] if c in geo_attrs.columns]]
master = skeleton.merge(geo_attrs,on="nis",how="left",validate="many_to_one")

pop_keep = [c for c in population.columns if c in [
    "year","nis","population","population_male","population_female","population_density_km2",
    "ln_population","ln_population_density","population_reference_date"
]]
master=merge_unique(master,population[pop_keep],label="population")
master=merge_unique(master,socio,label="socio")
master=merge_unique(master,employment.drop(columns=["area_km2"],errors="ignore"),label="employment")
master=merge_unique(master,education_wide,label="education")
master=merge_unique(master,sex_wide,label="sex")
master=merge_unique(master,status_wide,label="status")
master=merge_unique(master,sector_summary_wide,label="sector")
master=merge_unique(master,resnet,label="residence network")
master=merge_unique(master,worknet,label="workplace network")
master=merge_unique(master,accessibility,label="accessibility")
master=merge_unique(master,exposure_panel,label="telework")
master=merge_unique(master,network_exposure,label="network exposure")
master=merge_unique(master,spatial_exposure,label="spatial exposure")

# Common baseline/pre-period versions. These are created for exploration and are explicitly tagged later.
BASELINE_CANDIDATES = [c for c in [
    "ln_population","ln_population_density","ln_housing_cost_main",
    "ln_taxable_income_per_resident","ln_net_income_per_resident","ln_adi_median_wapprox",
    "ln_employment_density_km2",
    "resident_education_high_share","workplace_education_high_share",
    "resident_sector_hhi_known","workplace_sector_hhi_known",
    "job_access_exp50","population_access_exp50",
] if c in master.columns]

for c in BASELINE_CANDIDATES:
    b = master.loc[master.year.eq(BASELINE_YEAR),["nis",c]].rename(columns={c:f"{c}_2019"})
    master=master.merge(b,on="nis",how="left",validate="many_to_one")
    pre = master.loc[master.year.isin(PRE_YEARS)].groupby("nis",as_index=False)[c].mean().rename(
        columns={c:f"{c}_premean_2015_2019"})
    master=master.merge(pre,on="nis",how="left",validate="many_to_one")

# One-year lags for a compact set of time-varying covariates.
LAG_VARS=[c for c in [
    "ln_population","ln_population_density","ln_housing_cost_main",
    "ln_taxable_income_per_resident","ln_net_income_per_resident","ln_adi_median_wapprox",
    "ln_employment_density_km2","job_access_exp50",
] if c in master.columns]
lag=master[["nis","year"]+LAG_VARS].copy()
lag["year"]+=1
lag=lag.rename(columns={c:"lag1_"+c for c in LAG_VARS})
master=master.merge(lag,on=["nis","year"],how="left",validate="one_to_one")

master["post_2020"]=master.year.ge(SHOCK_YEAR).astype("int8")
master["event_time_2020"]=master.year-SHOCK_YEAR
master=master.sort_values(["nis","year"]).reset_index(drop=True)

require(len(master)==EXPECTED_N*len(YEARS),"Municipality master is not 565 x 10")
require(master.groupby("year").nis.nunique().eq(EXPECTED_N).all(),"A year has fewer than 565 municipalities")

write_table(master,OUT/"01_municipality_year_master.parquet")
if WRITE_MUNICIPALITY_CSV:
    master.to_csv(OUT/"01_municipality_year_master.csv",index=False,encoding="utf-8-sig")

if WRITE_GEOJSON_2024:
    a=master[master.year.eq(2024)].copy()
    g=geo[["nis","geometry"]].merge(a,on="nis",how="left",validate="one_to_one")
    g=gpd.GeoDataFrame(g,geometry="geometry",crs=geo.crs)
    g.to_file(OUT/"municipality_master_2024.geojson",driver="GeoJSON")

print("Municipality master:",master.shape)


## 14. Build the OD pair × year master for network/PPML analysis

In [ ]:

OD_NODE_VARS=[c for c in [
    "population","ln_population","population_density_km2","ln_population_density",
    "housing_cost_main","ln_housing_cost_main","taxable_income_per_resident","ln_taxable_income_per_resident",
    "adi_median_wapprox","ln_adi_median_wapprox","employment_density_km2","ln_employment_density_km2",
    "resident_education_high_share","workplace_education_high_share",
    "tw_res_current","tw_work_current","bartik_res_2019","bartik_work_2019",
    "network_out_tw_work_current_stable","network_in_tw_res_current_stable",
    "spatial_contig_tw_res_current","spatial_contig_tw_work_current",
] if c in master.columns]

o = master[["year","nis"]+OD_NODE_VARS].rename(columns={
    "nis":"home_nis", **{c:"o_"+c for c in OD_NODE_VARS}
})
d = master[["year","nis"]+OD_NODE_VARS].rename(columns={
    "nis":"work_nis", **{c:"d_"+c for c in OD_NODE_VARS}
})

od_master = od_domestic.merge(o,on=["year","home_nis"],how="left",validate="many_to_one")
od_master = od_master.merge(d,on=["year","work_nis"],how="left",validate="many_to_one")
od_master["post_2020"]=od_master.year.ge(SHOCK_YEAR).astype("int8")
od_master["event_time_2020"]=od_master.year-SHOCK_YEAR
od_master["ln_distance_plus1"]=np.log1p(od_master["distance_km"])

if {"o_tw_res_current","d_tw_work_current"}.issubset(od_master.columns):
    od_master["tw_pair_mean"]=(od_master["o_tw_res_current"]+od_master["d_tw_work_current"])/2
    od_master["tw_gap_work_minus_res"]=od_master["d_tw_work_current"]-od_master["o_tw_res_current"]
    od_master["distance_x_tw_work"]=od_master["distance_km"]*od_master["d_tw_work_current"]
    od_master["distance_x_tw_res"]=od_master["distance_km"]*od_master["o_tw_res_current"]
if {"o_bartik_res_2019","d_bartik_work_2019"}.issubset(od_master.columns):
    od_master["bartik_pair_mean"]=(od_master["o_bartik_res_2019"]+od_master["d_bartik_work_2019"])/2
    od_master["bartik_gap_work_minus_res"]=od_master["d_bartik_work_2019"]-od_master["o_bartik_res_2019"]

write_table(od_master,OUT/"02_od_pair_year_master.parquet")
if WRITE_OD_CSV_GZ:
    od_master.to_csv(OUT/"02_od_pair_year_master.csv.gz",index=False,encoding="utf-8-sig",compression="gzip")

if CREATE_BALANCED_PAIR_SKELETON:
    sk = pd.MultiIndex.from_product([YEARS,id_order,id_order],names=["year","home_nis","work_nis"]).to_frame(index=False)
    sk = sk.merge(pair,on=["home_nis","work_nis"],how="left",validate="many_to_one")
    obs = od_domestic[["year","home_nis","work_nis","workers","observed_in_source"]]
    sk = sk.merge(obs,on=["year","home_nis","work_nis"],how="left",validate="one_to_one")
    sk["observed_in_source"]=sk["observed_in_source"].fillna(False)
    sk["flow_missing_unpublished"]=sk["workers"].isna()
    if TREAT_UNPUBLISHED_OD_AS_ZERO:
        warnings.warn("You explicitly chose to treat unpublished OD cells as zero. This is an assumption, not a property verified from VAR.")
        sk["workers_assuming_unpublished_zero"]=sk["workers"].fillna(0)
    write_table(sk,SUPPORT/"od_balanced_pair_year_skeleton.parquet")
    print("Balanced pair skeleton:",sk.shape)

print("Observed OD master:",od_master.shape)


## 15. Variable roles, QA, source manifest and final exports

In [ ]:

def variable_role(c):
    lc=c.lower()
    if c in ("nis","year","node_id","var_name","home_nis","work_nis","pair_id"):
        return "identifier"
    if lc.startswith("network_") and ("tw_" in lc or "bartik" in lc):
        return "network_exposure"
    if lc.startswith("spatial_") and ("tw_" in lc or "bartik" in lc):
        return "spatial_exposure"
    if "bartik" in lc or lc.startswith("tw_") or "telework" in lc:
        return "treatment_exposure"
    if "_2019" in lc or "_premean_2015_2019" in lc:
        return "baseline_control"
    if lc.startswith("lag1_"):
        return "lagged_control"
    if lc.startswith("resnet_") or lc.startswith("worknet_"):
        return "outcome_candidate"
    if "coverage" in lc or "gap_share" in lc or lc.endswith("_approximation"):
        return "qa"
    if any(k in lc for k in ["housing","income","population","employment_density","access_"]):
        return "time_varying_control_or_mechanism"
    return "descriptive_or_supporting"

def post_treatment_note(c,role):
    if role=="time_varying_control_or_mechanism":
        return "May be affected by the 2020 shock; prefer pre-treatment baseline or lagged form in causal specifications unless a design justifies contemporaneous use."
    if role in ("network_exposure","spatial_exposure"):
        return "Exposure propagated through pre-specified weights where available; inspect definition before causal use."
    if role=="outcome_candidate":
        return "Network-derived outcome/descriptor; do not include contemporaneously as a control for the same network outcome."
    return ""

dictionary = pd.DataFrame({
    "variable":master.columns,
    "role":[variable_role(c) for c in master.columns],
})
dictionary["post_treatment_note"]=[post_treatment_note(c,r) for c,r in zip(dictionary.variable,dictionary.role)]
dictionary.to_csv(OUT/"10_variable_dictionary.csv",index=False,encoding="utf-8-sig")

coverage=[]
for c in master.columns:
    if c in ("nis","year"):
        continue
    q=master.groupby("year")[c].apply(lambda s:s.notna().mean()).reset_index(name="share_nonmissing")
    q["variable"]=c
    coverage.append(q)
coverage=pd.concat(coverage,ignore_index=True)
coverage.to_csv(OUT/"11_data_coverage_QA.csv",index=False,encoding="utf-8-sig")

qa_rows=[
    {"check":"municipality_master_rows","value":len(master),"expected":EXPECTED_N*len(YEARS),"pass":len(master)==EXPECTED_N*len(YEARS)},
    {"check":"municipalities_each_year","value":int(master.groupby("year").nis.nunique().min()),"expected":EXPECTED_N,"pass":master.groupby("year").nis.nunique().eq(EXPECTED_N).all()},
    {"check":"pair_static_rows","value":len(pair),"expected":EXPECTED_N**2,"pass":len(pair)==EXPECTED_N**2},
    {"check":"observed_od_negative_flows","value":int((od_master.workers<0).sum()),"expected":0,"pass":not (od_master.workers<0).any()},
]
if not exposure_panel.empty:
    rate_like = [
        c for c in exposure_panel.columns
        if c.startswith("tw_")
        and "workers" not in c
        and "coverage" not in c
        and (
            "current" in c
            or "frozen" in c
            or re.search(r"_s20\\d{2}_r20\\d{2}$", c)
        )
    ]
    if rate_like:
        bad=0
        for c in rate_like:
            s=pd.to_numeric(exposure_panel[c],errors="coerce").dropna()
            bad += int(((s<0)|(s>1)).sum())
        qa_rows.append({"check":"telework_rate_like_exposure_outside_0_1","value":bad,"expected":0,"pass":bad==0})
qa=pd.DataFrame(qa_rows)
qa.to_csv(QA_DIR/"master_integrity_checks.csv",index=False,encoding="utf-8-sig")

sources = {
    "basemap":BASEMAP,"population_raw":POP_XLSX,"population_existing":POP_PANEL_EXISTING,
    "housing":HOUSE_XLSX,"tax":TAX_XLSX,"adi":ADI_XLSX,
    **{"od_"+k:v for k,v in OD_FILES.items()},
    "var_direct_root":VAR_DIRECT_ROOT,"telework_original":TELEWORK_XLSX,
    "wfh_rate_component":WFH_RATE_CSV,"wfh_workplace_margin_component":WFH_WORKPLACE_MARGIN_CSV,
}
manifest=[]
for name,p in sources.items():
    p=Path(p)
    manifest.append({
        "source":name,"path":str(p),"exists":p.exists(),
        "size_mb":round(p.stat().st_size/1024**2,3) if p.is_file() else np.nan,
        "note":"directory" if p.is_dir() else ""
    })
pd.DataFrame(manifest).to_csv(OUT/"12_source_manifest.csv",index=False,encoding="utf-8-sig")

readme = f"""Belgium master data preparation v1
Generated from: {ROOT}
Years: {YEAR_MIN}-{YEAR_MAX}
Canonical geography: {EXPECTED_N} municipalities, boundary reference 2025-01-01

Main outputs
01_municipality_year_master.parquet
02_od_pair_year_master.parquet

Supporting outputs include pair-static distance/contiguity, residence/workplace network metrics,
telework exposures, pre-2019 commuting weights, spatial weights, sector panels, VAR direct archive inventory,
coverage QA and variable roles.

Important:
- Missing/unpublished OD cells are NOT changed to zero by default.
- Municipality self-links have representative-point distance 0; this is not an estimate of within-municipality travel distance.
- Current housing, income, employment, accessibility and network characteristics may be post-treatment variables after 2020.
- Baseline 2019 and 2015-2019 pre-period means are supplied for causal designs.
- Full VAR marginal tables are indexed but are not cross-joined into fictitious joint distributions.
"""
(OUT/"README_master_data.txt").write_text(readme,encoding="utf-8")

display(qa)
print("Saved:")
for p in [
    OUT/"01_municipality_year_master.parquet",
    OUT/"02_od_pair_year_master.parquet",
    OUT/"10_variable_dictionary.csv",
    OUT/"11_data_coverage_QA.csv",
    OUT/"12_source_manifest.csv",
]:
    print(" ",p)


## 16. Optional quick exploration checks

In [ ]:

# These plots are descriptive QA only.
plt.rcParams.update({"font.family":"serif","font.serif":["Times New Roman","DejaVu Serif"],
                     "axes.spines.top":False,"axes.spines.right":False})

fig,ax=plt.subplots(figsize=(7.2,4.2))
annual=od_domestic.groupby("year")["workers"].sum()
ax.plot(annual.index,annual.values,marker="o")
ax.set(xlabel="Year",ylabel="Published domestic worker flow",title="Belgian job–home network coverage over time")
fig.tight_layout()
plt.show()

expcols=[c for c in ["tw_res_current","tw_work_current","bartik_res_2019","bartik_work_2019"] if c in master.columns]
if expcols:
    display(master.groupby("year")[expcols].agg(["mean","std","count"]).round(4))

display(master[["year","nis"]+[c for c in [
    "resnet_mean_km","resnet_external_mean_km","resnet_cross_share",
    "worknet_mean_km","worknet_external_mean_km","worknet_cross_share"
] if c in master.columns]].head())
